In [3]:
!pip install pandas sqlalchemy psycopg2-binary python-dotenv

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 2.9 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 3.5 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.8 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 3.6 MB/s eta 0:00:00


In [4]:
import os
import sqlite3
import pandas as pd
import psycopg2

from dotenv import load_dotenv
from sqlalchemy import create_engine

# -----------------------------------
# Load environment variables
# -----------------------------------

load_dotenv()

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DB = os.getenv("POSTGRES_DB")

SQLITE_DB_PATH = os.getenv("SQLITE_DB_PATH")

# -----------------------------------
# SQLite connection
# -----------------------------------

sqlite_conn = sqlite3.connect(SQLITE_DB_PATH)

# -----------------------------------
# PostgreSQL connection
# -----------------------------------

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}"
    f"/{POSTGRES_DB}"
)

postgres_engine = create_engine(DATABASE_URL)

# -----------------------------------
# Get SQLite tables
# -----------------------------------

tables_query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(tables_query, sqlite_conn)

print("\nTables found:")
print(tables)

# -----------------------------------
# Transfer tables
# -----------------------------------

for table in tables['name']:

    print(f"\nTransferring table: {table}")

    df = pd.read_sql(f"SELECT * FROM {table}", sqlite_conn)

    df.to_sql(
        table,
        postgres_engine,
        if_exists='replace',
        index=False
    )

    print(f"Done: {table}")

print("\nMigration completed successfully.")


Tables found:
              name
0        purchases
1  purchase_prices
2   vendor_invoice
3  begin_inventory
4    end_inventory

Transferring table: purchases
Done: purchases

Transferring table: purchase_prices
Done: purchase_prices

Transferring table: vendor_invoice
Done: vendor_invoice

Transferring table: begin_inventory
Done: begin_inventory

Transferring table: end_inventory
Done: end_inventory

Migration completed successfully.
